## Problem: Numerical Stability and Subtractive Cancellation

In this problem, we study the effect of **finite-precision arithmetic** and the phenomenon of **catastrophic cancellation** by comparing two mathematically equivalent expressions.

We are given the following two functions:

$$
E_1(x) = \frac{1 - \cos x}{\sin^2 x}, \qquad
E_2(x) = \frac{1}{1 + \cos x}
$$

Using trigonometric identities, we have:

$$
\sin^2 x = 1 - \cos^2 x = (1 - \cos x)(1 + \cos x)
$$

Therefore,

$$
\begin{aligned}
E_1(x) &= \frac{1 - \cos x}{\sin^2 x} \\
       &= \frac{1 - \cos x}{(1 - \cos x)(1 + \cos x)} \\
       &= \frac{1}{1 + \cos x} \\
       &= E_2(x)
\end{aligned}
$$

So, **\(E_1(x)\) and \(E_2(x)\) are mathematically identical**.

---

### Objective

1. Compute the values of \(E_1(x)\) and \(E_2(x)\) for the following values of \(x\):

   $$
   x = 10^{0},\, 10^{-1},\, 10^{-2},\, \dots,\, 10^{-8}
   $$

2. Tabulate the results in a table with three columns: \(x\), \(E_1(x)\), and \(E_2(x)\).

3. Compare the numerical values of \(E_1\) and \(E_2\) for small \(x\), and discuss:

   - Why the results of \(E_1(x)\) and \(E_2(x)\) start to differ in floating-point arithmetic.
   - Which expression is **more numerically stable** and why.
   - How **subtractive cancellation** appears in the computation of \(E_1(x)\) when \(x\) is very small.



In [28]:
import numpy as np
from tabulate import tabulate

In [13]:
def E1(x):
    return (1 - np.cos(x)) / np.sin(x)**2

In [14]:
def E2(x):
    return 1 / (1 + np.cos(x))

In [15]:
x = [10 ** (-i) for i in range(9)]

In [18]:
E1_x = [E1(i) for i in x]
E2_x = [E2(i) for i in x]

In [27]:
table = []

for i in range(len(x)):
    row = [x[i], E1_x[i], E2_x[i]]
    table.append(row)

headers = ["x", "E1", "E2"]

print(tabulate(table, headers=headers, floatfmt=".15f"))

                x                 E1                 E2
-----------------  -----------------  -----------------
1.000000000000000  0.649223205204762  0.649223205204762
0.100000000000000  0.501252086288566  0.501252086288571
0.010000000000000  0.500012500208481  0.500012500208336
0.001000000000000  0.500000124992189  0.500000125000021
0.000100000000000  0.499999998627931  0.500000001250000
0.000010000000000  0.500000041386852  0.500000000012500
0.000001000000000  0.500044450291337  0.500000000000125
0.000000100000000  0.499600361081322  0.500000000000001
0.000000010000000  0.000000000000000  0.500000000000000


### Analysis of Results: Subtractive Cancellation in Action

Based on the output table, we can observe the following critical points regarding finite-precision arithmetic:

1. **Theoretical Limit:**
   As $x \to 0$, the value of $\cos(x) \to 1$. Therefore, mathematically, the limit for both expressions is:
   $$ \lim_{x \to 0} E_1(x) = \lim_{x \to 0} E_2(x) = \frac{1}{1+1} = 0.5 $$

2. **Numerical Stability of $E_2$ (The Stable Form):**
   The column for $E_2$ behaves exactly as expected mathematically. As $x$ decreases, $E_2$ smoothly and consistently approaches $0.5$. Since the operation in the denominator is $1 + \cos(x)$, we are adding two positive numbers of similar magnitude ($1 + \approx 1 \approx 2$). Addition of positive numbers does not suffer from precision loss, making $E_2$ perfectly stable.

3. **Catastrophic Cancellation in $E_1$ (The Unstable Form):**
   The column for $E_1$ clearly demonstrates severe numerical instability:
   * For $x = 1.0$ and $0.1$, $E_1$ matches $E_2$ very closely.
   * As $x$ gets smaller (e.g., $x = 10^{-4}$ to $10^{-7}$), $E_1$ starts producing garbage digits (e.g., $0.49999...$, $0.50004...$, $0.4996...$). This is because we are losing significant digits when subtracting two numbers that are very close to each other.
   * **The Complete Breakdown:** At $x = 10^{-8}$, $E_1$ abruptly drops to exactly `0.000000000000000`.

4. **Why did $E_1$ become exactly zero?**
   For $x = 10^{-8}$, the true value of $\cos(10^{-8})$ is extremely close to $1$. In the 64-bit floating-point precision standard (IEEE 754) used by Python, there are not enough bits to store the extremely tiny difference between $1$ and $\cos(10^{-8})$. Consequently, the computer rounds $\cos(10^{-8})$ to exactly $1.0$. 
   Thus, the numerator operation becomes:
   $$ 1 - \cos(10^{-8}) \approx 1 - 1.0 = 0.0 $$
   This leads to $0 / \sin^2(10^{-8}) = 0$. This specific phenomenon is the textbook definition of **Subtractive (or Catastrophic) Cancellation**.

**Conclusion:**
In numerical methods and algorithm design, expressions involving the subtraction of nearly equal numbers (like $1 - \cos x$ for small $x$) must be avoided. Rewriting the expression into a mathematically equivalent form that avoids this subtraction (such as $E_2$) is essential for ensuring numerical stability.
